In [1]:
from pathlib import Path
import numpy as np

# Input kinematics (time + joint columns)
input_csv = Path("general_swim_extended_sine_fit.csv")
output_dir = Path("kinematics")
output_dir.mkdir(parents=True, exist_ok=True)

# Target frequencies (Hz)
# Use raw data directly for 4.5 Hz as requested.
freqs = [2.5, 3.0, 3.5, 4.0, 4.5, 5.0]

raw = np.loadtxt(input_csv, delimiter=",")
t = raw[:, 0]
X = raw[:, 1:]

# Estimate nominal frequency from dominant FFT peak across all columns.
dt = float(np.mean(np.diff(t)))
fft_freq = np.fft.rfftfreq(len(t), d=dt)

dom = []
for j in range(X.shape[1]):
    y = X[:, j] - X[:, j].mean()
    Y = np.abs(np.fft.rfft(y))
    k = 1 + np.argmax(Y[1:])
    dom.append(fft_freq[k])
f_raw = float(np.median(dom))

# Build per-signal phase template over one cycle so amplitude is preserved.
phase_raw = np.mod(t * f_raw, 1.0)
bins = 2000
edges = np.linspace(0.0, 1.0, bins + 1)
centers = 0.5 * (edges[:-1] + edges[1:])

def build_template(y: np.ndarray) -> np.ndarray:
    inds = np.digitize(phase_raw, edges) - 1
    inds = np.clip(inds, 0, bins - 1)
    sums = np.zeros(bins)
    counts = np.zeros(bins)
    np.add.at(sums, inds, y)
    np.add.at(counts, inds, 1)

    template = np.empty(bins)
    valid = counts > 0
    template[valid] = sums[valid] / counts[valid]

    # Fill any empty bins by linear interpolation in phase.
    if not np.all(valid):
        template[~valid] = np.interp(centers[~valid], centers[valid], template[valid])

    return template

templates = [build_template(X[:, j]) for j in range(X.shape[1])]

# Keep same time vector length/resolution; only change oscillation frequency.
t_out = t.copy()

for f_new in freqs:
    if abs(f_new - 4.5) < 1e-12:
        out = raw.copy()
    else:
        phase_new = np.mod(t_out * f_new, 1.0)
        out_cols = [
            np.interp(phase_new, centers, tpl, period=1.0)
            for tpl in templates
        ]
        out = np.column_stack([t_out] + out_cols)

    out_path = output_dir / f"nominal_swim_{f_new:.1f}.csv"
    np.savetxt(out_path, out, delimiter=",")

print(f"Input file: {input_csv}")
print(f"Estimated raw dominant frequency: {f_raw:.6f} Hz")
print("Written files:")
for f_new in freqs:
    print(output_dir / f"nominal_swim_{f_new:.1f}.csv")

Input file: general_swim_extended_sine_fit.csv
Estimated raw dominant frequency: 4.587156 Hz
Written files:
kinematics/nominal_swim_2.5.csv
kinematics/nominal_swim_3.0.csv
kinematics/nominal_swim_3.5.csv
kinematics/nominal_swim_4.0.csv
kinematics/nominal_swim_4.5.csv
kinematics/nominal_swim_5.0.csv
